# Tu primer modelo de ML aplicado a construcción

En este notebook construiremos un modelo completo de principio a fin: desde cargar datos hasta evaluar predicciones. Usaremos un dataset simulado de proyectos de edificación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print("Librerías importadas ✓")

## Paso 1: Crear el dataset

En la práctica, tus datos vendrán de hojas de cálculo, bases de datos o modelos BIM. Aquí generamos un dataset realista de 300 proyectos de edificación en Colombia.

In [ ]:
np.random.seed(42)
n = 300

# Generar características de proyectos
datos = pd.DataFrame({
    'area_m2':          np.random.uniform(200, 5000, n),
    'pisos':            np.random.randint(1, 30, n),
    'sotanos':          np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.35, 0.25, 0.1]),
    'tipo_estructura':  np.random.choice(['porticos', 'muros', 'dual'], n, p=[0.3, 0.4, 0.3]),
    'zona_sismica':     np.random.choice(['baja', 'intermedia', 'alta'], n, p=[0.2, 0.3, 0.5]),
    'ciudad':           np.random.choice(['bogota', 'medellin', 'cali', 'barranquilla'], n),
})

# Generar costo (variable objetivo) con relaciones realistas
base = datos['area_m2'] * 3.2  # ~3.2M COP/m²
factor_pisos = 1 + (datos['pisos'] - 1) * 0.015  # +1.5% por piso adicional
factor_sotanos = 1 + datos['sotanos'] * 0.08  # +8% por sótano
factor_estructura = datos['tipo_estructura'].map({'porticos': 1.0, 'muros': 1.15, 'dual': 1.10})
factor_sismo = datos['zona_sismica'].map({'baja': 1.0, 'intermedia': 1.05, 'alta': 1.12})
factor_ciudad = datos['ciudad'].map({'bogota': 1.0, 'medellin': 0.95, 'cali': 0.92, 'barranquilla': 0.88})

datos['costo_total_mcop'] = (
    base * factor_pisos * factor_sotanos * factor_estructura * factor_sismo * factor_ciudad
    + np.random.normal(0, 200, n)  # ruido
)

print(f"Dataset: {datos.shape[0]} proyectos, {datos.shape[1]} columnas")
datos.head(10)

## Paso 2: Exploración de datos (EDA)

Antes de modelar, **siempre** explora tus datos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Distribución de costos
axes[0, 0].hist(datos['costo_total_mcop'], bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribución de Costos')
axes[0, 0].set_xlabel('Costo total (millones COP)')

# Costo vs Área
axes[0, 1].scatter(datos['area_m2'], datos['costo_total_mcop'], alpha=0.5, s=20, c='steelblue')
axes[0, 1].set_title('Costo vs Área')
axes[0, 1].set_xlabel('Área (m²)')
axes[0, 1].set_ylabel('Costo (M COP)')

# Costo por tipo de estructura
datos.boxplot(column='costo_total_mcop', by='tipo_estructura', ax=axes[1, 0])
axes[1, 0].set_title('Costo por tipo de estructura')
axes[1, 0].set_xlabel('Tipo')
axes[1, 0].set_ylabel('Costo (M COP)')

# Costo por zona sísmica
datos.boxplot(column='costo_total_mcop', by='zona_sismica', ax=axes[1, 1])
axes[1, 1].set_title('Costo por zona sísmica')
axes[1, 1].set_xlabel('Zona')
axes[1, 1].set_ylabel('Costo (M COP)')

plt.suptitle('')  # Quitar título automático de pandas
plt.tight_layout()
plt.show()

## Paso 3: Preparar datos para el modelo

In [ ]:
# Convertir variables categóricas a numéricas (one-hot encoding)
datos_modelo = pd.get_dummies(datos, columns=['tipo_estructura', 'zona_sismica', 'ciudad'], drop_first=True)

# Separar características (X) y variable objetivo (y)
X = datos_modelo.drop('costo_total_mcop', axis=1)
y = datos_modelo['costo_total_mcop']

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Entrenamiento: {X_train.shape[0]} proyectos")
print(f"Prueba:        {X_test.shape[0]} proyectos")
print(f"\nVariables:     {list(X.columns)}")

## Paso 4: Entrenar el modelo

Usaremos **Random Forest** — un modelo robusto que funciona bien con datos tabulares y no requiere mucha preparación.

In [ ]:
# Entrenar Random Forest
modelo = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)

# Predecir
y_pred = modelo.predict(X_test)

# Evaluar
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Resultados del modelo:")
print(f"  Error Absoluto Medio (MAE): {mae:,.0f} millones COP")
print(f"  R² Score:                   {r2:.4f}")
print(f"\nInterpretación:")
print(f"  El modelo predice el costo con un error promedio de {mae:,.0f}M COP")
print(f"  y explica el {r2*100:.1f}% de la variabilidad en los costos.")

In [ ]:
# --- Visualización: Predicho vs Real ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=50)
lim = [min(y_test.min(), y_pred.min()) * 0.9, max(y_test.max(), y_pred.max()) * 1.1]
axes[0].plot(lim, lim, 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Costo real (M COP)')
axes[0].set_ylabel('Costo predicho (M COP)')
axes[0].set_title(f'Predicción vs Realidad (R² = {r2:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Importancia de variables
importancia = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=True)
importancia.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Importancia de cada variable')
axes[1].set_xlabel('Importancia relativa')

plt.tight_layout()
plt.show()

## Paso 5: Usar el modelo para predecir

Ahora puedes predecir el costo de un proyecto nuevo.

In [ ]:
# --- Predecir costo de un proyecto nuevo ---

# Definir características del nuevo proyecto
nuevo_proyecto = {
    'area_m2': 1500,
    'pisos': 12,
    'sotanos': 2,
    'tipo_estructura': 'dual',
    'zona_sismica': 'alta',
    'ciudad': 'bogota',
}

# Preparar datos (mismo formato que el entrenamiento)
nuevo_df = pd.DataFrame([nuevo_proyecto])
nuevo_encoded = pd.get_dummies(nuevo_df, columns=['tipo_estructura', 'zona_sismica', 'ciudad'], drop_first=True)

# Asegurar que tenga las mismas columnas que el modelo
for col in X.columns:
    if col not in nuevo_encoded.columns:
        nuevo_encoded[col] = 0
nuevo_encoded = nuevo_encoded[X.columns]

# Predecir
costo_estimado = modelo.predict(nuevo_encoded)[0]

print("ESTIMACIÓN DE COSTO")
print("=" * 40)
for k, v in nuevo_proyecto.items():
    print(f"  {k}: {v}")
print("=" * 40)
print(f"  Costo estimado: {costo_estimado:,.0f} millones COP")
print(f"  Costo por m²:   {costo_estimado/1500*1e6:,.0f} COP/m²")

## Resumen del flujo ML

```
Datos (Excel/BIM/BD)
    │
    ▼
Exploración (EDA)
    │
    ▼
Preparación (encoding, split)
    │
    ▼
Entrenamiento (Random Forest)
    │
    ▼
Evaluación (MAE, R²)
    │
    ▼
Predicción (nuevo proyecto)
```

## Ejercicios propuestos

1. **Agrega una variable**: Incluye `tipo_suelo` (arcilla, limo, roca) y observa si mejora el modelo.
2. **Cambia el modelo**: Prueba con `GradientBoostingRegressor` en lugar de `RandomForestRegressor`. ¿Mejora?
3. **Datos reales**: Reemplaza el dataset simulado con datos de proyectos reales de tu empresa.
4. **Exportar modelo**: Usa `joblib.dump(modelo, 'modelo_costos.pkl')` para guardarlo y reutilizarlo.